# Day 2 — Autoregressive Models & Supervised Learning Framing
## AR Models, Stability, and Lag-Embedded Feature Matrices

**Student Learning Outcomes:**
> **SLO 2:** Construct and interpret autoregressive (AR) models, including deriving and
> explaining the stability condition for an AR(1) process.
>
> **SLO 3:** Transform a time series into a supervised learning dataset by building
> lag-embedded feature matrices for univariate and multivariate data.


## Before We Begin

> **Prompt:** Think about something you do on autopilot — a habit, a routine, a reflex.
> How much does your *past behavior* predict your *next action*?
>
> Write 3–5 sentences describing this habit. Then draw a simple diagram showing how one moment leads to the next.


---
### Quick Recap from Day 1

Yesterday we learned to *diagnose* a time series: check for stationarity, apply
transformations, and read ACF/PACF plots.

Today we ask: **how do we actually model the dependence structure we found?**

In this notebook you will:
1. Understand what an autoregressive (AR) model is and how it works
2. Simulate AR(1) processes and observe how the parameter $\phi$ controls behavior
3. Derive and apply the stability (stationarity) condition for AR(1)
4. Fit an AR model to real data using `statsmodels`
5. Re-frame a time series as a supervised learning problem using **lag-embedded feature matrices**
6. Build lag matrices for both univariate and multivariate time series


---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.


In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Time series tools
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Reproducibility
np.random.seed(42)

# Display settings
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')


---
## Part 1 — What is an Autoregressive Model?

### 1.1 The AR(1) model

An **autoregressive model of order 1**, written **AR(1)**, says that the current value
of a series depends linearly on the *previous* value plus some random noise:

$$x_t = \phi \, x_{t-1} + \varepsilon_t$$

where:
- $x_t$ is the value at time $t$
- $\phi$ (phi) is the **autoregressive coefficient** — it controls how strongly the past
  influences the present
- $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$ is white noise (random, mean-zero error)

More generally, an **AR(p)** model uses the last $p$ values:

$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + \cdots + \phi_p x_{t-p} + \varepsilon_t$$

### 1.2 Simulating AR(1) processes

Before fitting a model to real data, it helps to *simulate* the process and see how
different values of $\phi$ produce very different behaviors.

We will complete the cell below together.


In [ ]:
def simulate_ar1(phi, n=200, sigma=1.0, x0=0.0):
    """Simulate an AR(1) process: x[t] = phi * x[t-1] + noise."""
    x = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        x[t] = ???
    return x

phi_values = [0.0, 0.5, 0.9, 0.99, 1.0, 1.05, -0.7]
labels     = [
    'phi=0.0 (white noise)',
    'phi=0.5 (moderate memory)',
    'phi=0.9 (strong memory)',
    'phi=0.99 (near unit root)',
    'phi=1.0 (random walk)',
    'phi=1.05 (explosive)',
    'phi=-0.7 (oscillating)',
]

# create a for-loop to plot the different values of phi

### ✏️ Written Response 1.2

Study the seven simulated series above and answer:

1. What happens to the series as $\phi$ increases from 0 toward 1? Describe the change in
   behavior in your own words.
2. What is qualitatively different about $\phi = 1.0$ versus $\phi = 0.99$?
3. What does $\phi = 1.05$ do? Why might this be problematic for a model?
4. What does a **negative** $\phi$ produce? Why does the series oscillate?

> **YOUR ANSWER:**


---
## Part 2 — The Stability Condition for AR(1)

### 2.1 Why does $|\phi| < 1$ matter?

From your simulations above, you can see that the AR(1) process behaves very differently
depending on $\phi$. The **stability condition** for an AR(1) process is:

$$|\phi| < 1$$

When this holds, the process is **stationary** — it has a well-defined, constant mean and
variance over time. When $|\phi| \geq 1$, the process is non-stationary (it either
drifts like a random walk or explodes).

### 2.2 Deriving the mean and variance of a stationary AR(1)

If $x_t = \phi x_{t-1} + \varepsilon_t$ is stationary, then by definition the mean
$\mu = \mathbb{E}[x_t]$ does not change over time:

$$\mu = \phi \mu + 0 \implies \mu(1 - \phi) = 0 \implies \mu = 0$$

(assuming zero-mean noise; if there's a constant term $c$, then $\mu = c / (1-\phi)$)

Similarly, the variance $\gamma_0 = \text{Var}(x_t)$ satisfies:

$$\gamma_0 = \frac{\sigma^2}{1 - \phi^2}$$

Notice: this only makes sense when $|\phi| < 1$. Otherwise, what happens?

### 2.3 Verify with simulation

**Your turn!** Fill in the cell below to compute the **theoretical variance** and compare
it to the **sample variance** from a simulation.


In [ ]:
phi   = 0.7
sigma = 1.0
n     = 5000

# FILL IN: theoretical variance for a stationary AR(1)
theoretical_var = ???

# Simulate and compute sample variance


print(f'phi             = {phi}')
print(f'Theoretical var = {theoretical_var:.4f}')
print(f'Sample var      = {sample_var:.4f}')
print(f'Difference      = {abs(theoretical_var - sample_var):.4f}')


### ✏️ Written Response 2.3

1. How close is the sample variance to the theoretical variance? What does this confirm?
2. **dividir y confluir:** Try changing `phi` to a different value (groups of 2 to 3). What happens to the theoretical variance? Why?
3. What happens if you try `phi = 1.0`? (what error or result do you get,
   and why does it make mathematical sense?)

> **YOUR ANSWER:**


---
## Part 3 — ACF Signature of AR Models

From Day 1, recall:
- **ACF** decays slowly for non-stationary series
- **PACF** cuts off at lag $p$ for an AR($p$) process

Let's verify this for AR(1) and AR(2) simulations.

### 3.1 ACF and PACF of simulated AR processes

Complete the cell below, then run it and study how the PACF behaves.


In [ ]:
def simulate_ar2(phi1, phi2, n=500, sigma=1.0):
    """Simulate an AR(2) process: x[t] = phi1*x[t-1] + phi2*x[t-2] + noise."""
    x = np.zeros(n)
    for t in range(2, n):
        x[t] = ???
    return x

ar1_series = simulate_ar1(phi=0.8, n=500)
ar2_series = simulate_ar2(phi1=0.6, phi2=0.3, n=500)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

plot_acf( ar1_series, lags=20, ax=axes[0, 0])
axes[0, 0].set_title('ACF — AR(1), phi=0.8')

plot_pacf(ar1_series, lags=20, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('PACF — AR(1), phi=0.8')

plot_acf( ar2_series, lags=20, ax=axes[1, 0])
axes[1, 0].set_title('ACF — AR(2), phi1=0.6, phi2=0.3')

plot_pacf(ar2_series, lags=20, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('PACF — AR(2), phi1=0.6, phi2=0.3')

plt.tight_layout()
plt.show()


### ✏️ Written Response 3.1

1. In the AR(1) PACF, how many lags are significant (outside the blue band)?
   What does this tell you about the order of the process?
2. In the AR(2) PACF, how many lags are significant? How does this differ from AR(1)?
3. The ACF for both processes shows a gradual decay. Why isn't the ACF as useful as
   the PACF for identifying the *order* of an AR model?

> **YOUR ANSWER:**


---
## Part 4 — Fitting an AR Model to Real Data

Now let's load the Air Passengers data again and fit an AR model. Because the raw series
is non-stationary, we'll work with the **log-differenced** version from Day 1.

### 4.1 Load and prepare the data

This cell below is complete.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']

# Apply the same transformations from Day 1
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff'] = df['Log_Passengers'].diff()

# Drop the NaN introduced by differencing
series = df['Log_Diff'].dropna()

print('Series length:', len(series))
series.head()


### 4.2 Choose the AR order using the PACF

**Your turn!** Plot the PACF of `series` to decide what order AR model to fit.
Use at most 24 lags (2 years of monthly data).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# FILL IN: plot ACF of series with 24 lags

# FILL IN: plot PACF of series with 24 lags

plt.tight_layout()
plt.show()


> 💡 **Hint:** Look at where the PACF spikes fall *outside* the blue confidence band.
> The number of significant lags suggests the AR order $p$.


### 4.3 Fit the AR model

`statsmodels` provides `AutoReg` for fitting AR models. The key argument is `lags`,
which sets the order $p$.

> 💡 **Syntax reminder:** `AutoReg(series, lags=p).fit()` fits the model.
> `.params` gives the estimated coefficients. `.summary()` gives a full statistical summary.


In [ ]:
# FILL IN: choose your AR order based on the PACF plot above
p = ???

model = AutoReg(series, lags=p)
result = model.fit()

print(result.summary())


### 4.4 Inspect the fitted coefficients

**Your turn!** Extract and print the fitted $\phi$ coefficients.


In [ ]:
print('Fitted AR coefficients:')
print(result.params)

# FILL IN: check stability — are all |phi| < 1?
phi_estimates = result.params[1:]  # skip the intercept
print('\nAre all |phi| < 1?', all(abs(phi_estimates) < ???))


### ✏️ Written Response 4

1. What order $p$ did you choose based on the PACF, and why?
2. What are the estimated $\phi$ coefficients? Are they statistically significant
   (look at the p-values in the summary)?
3. Does the fitted model satisfy the stability condition $|\phi| < 1$?
4. How would you interpret the largest $\phi$ coefficient in plain language?

> **YOUR ANSWER:**


---
## Part 5 — Re-framing as a Supervised Learning Problem

### 5.1 The key idea

An AR($p$) model is essentially a **linear regression** where the features are lagged
values of the target:

$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + \cdots + \phi_p x_{t-p} + \varepsilon_t$$

This means we can turn any time series into a tabular dataset that *any* machine
learning model can consume — not just AR. This is the foundation for tree-based
models, MLPs, and other models we'll use later this week.

The transformation looks like this:

| $x_{t-2}$ | $x_{t-1}$ | $x_t$ (target) |
|---|---|---|
| $x_1$ | $x_2$ | $x_3$ |
| $x_2$ | $x_3$ | $x_4$ |
| $x_3$ | $x_4$ | $x_5$ |
| ⋮ | ⋮ | ⋮ |

Each row is one training example. The **lag columns** are features ($X$);
the current value is the **target** ($y$). This is called a **lag-embedded feature matrix**.

### 5.2 Build the lag matrix function

Below is the skeleton of a function that creates a lag-embedded feature matrix
from a univariate time series. **Fill in the missing lines.**

> 💡 **Syntax reminder:** `np.roll(arr, shift)` shifts an array by `shift` positions.
> `pd.DataFrame.shift(k)` shifts a Series by `k` time steps (introduces NaN at the start).


In [ ]:
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series  : array-like, shape (T,)
    n_lags  : int, number of lag features to create

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)   <- feature matrix
    y : np.ndarray, shape (T - n_lags,)           <- target vector
    """
    series = np.array(series)
    T = len(series)
    X, y = [], []

    for t in range(n_lags, T):
        # FILL IN: features are the previous n_lags values
        X.append(???)
        # FILL IN: target is the value at time t
        y.append(???)

    return np.array(X), np.array(y)


In [ ]:
# Test your function on the log-differenced series
X, y = make_lag_matrix(series.values, n_lags=3)

print('Feature matrix X shape:', X.shape)
print('Target vector y shape: ', y.shape)
print()
print('First 5 rows of X (lag features):')
print(X[:5])
print()
print('First 5 targets y:')
print(y[:5])


### 5.3 View as a tidy DataFrame

**Your turn!** Wrap the output of `make_lag_matrix` in a `pd.DataFrame` with
descriptive column names (`lag_1`, `lag_2`, ..., `lag_p`, `target`).


In [ ]:
n_lags = 3
X, y = make_lag_matrix(series.values, n_lags=n_lags)

# FILL IN: create column names ['lag_1', 'lag_2', 'lag_3', 'target']
col_names = [f'lag_{i}' for i in range(???, ???)] + ['???']

lag_df = pd.DataFrame(
    np.column_stack([X, y]),
    columns=col_names
)

print('Shape:', lag_df.shape)
lag_df.head(10)


### ✏️ Written Response 5.3

1. How many rows does the lag matrix have compared to the original series? Why?
2. In plain language, what does each *row* of the lag matrix represent?
3. If you used `n_lags=12` on monthly data, what would that mean conceptually?

> **YOUR ANSWER:**


---
## Part 6 — Multivariate Lag Matrices

The same idea extends to **multiple time series**. For example, you might want to
predict passenger counts using lagged values of *both* passenger counts and some
other variable (e.g., fuel prices, GDP, temperature).

We'll simulate a second series to demonstrate the concept.

### 6.1 Create a synthetic multivariate dataset

This cell is complete — run it.


In [ ]:
# Use log-differenced passengers as series 1
s1 = series.values

# Simulate a correlated second series (e.g., a noisy economic indicator)
np.random.seed(0)
s2 = 0.5 * s1 + np.random.normal(0, 0.02, size=len(s1))

# Pack into a DataFrame
multi_df = pd.DataFrame({'passengers_log_diff': s1, 'indicator': s2})
print('Shape:', multi_df.shape)
multi_df.head()


### 6.2 Build a multivariate lag matrix function

**Your turn!** Fill in the function below. The idea is the same as before, but now
you create lag columns for *every* variable in the DataFrame.

> 💡 **Syntax reminder:** `df.shift(k)` shifts every column of a DataFrame down by
> `k` rows (the first `k` rows become NaN). `df.dropna()` removes rows with any NaN.


In [ ]:
def make_lag_matrix_multi(df, n_lags, target_col):
    """
    Build a lag-embedded feature matrix from a multivariate time series.

    Parameters
    ----------
    df         : pd.DataFrame, shape (T, n_variables)
    n_lags     : int, number of lags per variable
    target_col : str, column name of the variable to forecast

    Returns
    -------
    X : pd.DataFrame of lag features
    y : pd.Series of targets
    """
    lagged_frames = []

    for lag in range(1, n_lags + 1):
        # FILL IN: shift the entire DataFrame by `lag` steps
        shifted = df.shift(???)
        # FILL IN: rename columns to indicate the lag, e.g. 'passengers_log_diff_lag1'
        shifted.columns = [f'{col}_lag{lag}' for col in df.columns]
        lagged_frames.append(???)

    # Combine all lagged frames side by side
    feature_df = pd.concat(lagged_frames, axis=1)

    # FILL IN: the target is the current (unshifted) target column
    target = df[???]

    # Drop rows where any lag is NaN (the first n_lags rows)
    combined = pd.concat([feature_df, target], axis=1).dropna()
    X = combined.drop(columns=[target_col])
    y = combined[target_col]

    return X, y


In [ ]:
# Test the multivariate function
X_multi, y_multi = make_lag_matrix_multi(
    df=multi_df,
    n_lags=3,
    target_col='passengers_log_diff'
)

print('Feature matrix shape:', X_multi.shape)
print('Target vector shape: ', y_multi.shape)
print()
print('Column names:')
print(list(X_multi.columns))
print()
X_multi.head()


### ✏️ Written Response 6.2

1. How many feature columns does the multivariate lag matrix have? How is this computed
   from the number of variables and the number of lags?
2. Why do we drop the first `n_lags` rows after building the lag matrix?
3. In what situation would adding a second (or third) variable's lags be useful?
   Give a real-world example.

> **YOUR ANSWER:**


---
## Part 7 — Putting It All Together

### 7.1 From AR to supervised learning: the connection

The lag matrix you built in Part 5 is *exactly* the input representation needed to
use *any* machine learning model for forecasting. Let's verify that a simple linear
regression on the lag matrix gives the same result as `AutoReg`.

This cell is complete — run it and compare the coefficients.


In [ ]:
from sklearn.linear_model import LinearRegression

# Build the lag matrix with the same order as your AR model above
X_lr, y_lr = make_lag_matrix(series.values, n_lags=p)

lr = LinearRegression(fit_intercept=True)
lr.fit(X_lr, y_lr)

print('LinearRegression coefficients (lag_1, lag_2, ...):')
print(lr.coef_)
print('Intercept:', lr.intercept_)
print()
print('AutoReg coefficients (intercept, lag_1, lag_2, ...):')
print(result.params.values)


### ✏️ Final Written Reflection

Write a **4–6 sentence summary** connecting today's topics as if explaining to a
classmate who missed Day 2. Your summary must address:

- What an AR($p$) model does and what the coefficients mean
- Why the stability condition $|\phi| < 1$ matters
- How you used the PACF to choose the order $p$
- How a lag-embedded feature matrix re-frames forecasting as supervised learning
- Why this re-framing is powerful (what models does it unlock?)

> **YOUR ANSWER:**


---
## Moving Forward

Choose a different time series (your own data or any publicly available dataset) and:

1. Run the `ts_diagnostics()` function from Day 1
2. Use the PACF to select an AR order
3. Fit an `AutoReg` model and check the stability condition
4. Build a lag matrix using `make_lag_matrix`
5. Fit a `LinearRegression` model on the lag matrix and compare coefficients

```python
# Your code here
```


## References / Further Reading

* [Forecasting: Principles and Practice — Chapter 9 (ARIMA)](https://otexts.com/fpp3/arima.html)
* [statsmodels AutoReg documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.ar_model.AutoReg.html)
* [Machine Learning for Time Series Forecasting — Géron](https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/)
